# EDA — Vehicle Dashboard

## Objetivo

Este notebook documenta a análise exploratória de dados (EDA) do conjunto de anúncios de veículos usados utilizado no projeto **Vehicle Dashboard**.

A análise inclui:
- inspeção inicial e estrutura dos dados;
- estatísticas descritivas;
- análise de variáveis categóricas;
- identificação de valores ausentes;
- investigação de valores extremos;
- análise de relações entre variáveis e `price`;
- tratamento dos dados;
- validação do dataset final utilizado pelo dashboard.


## 1. Importação das bibliotecas e leitura dos dados

Como o notebook está dentro da pasta `notebooks/`, o arquivo CSV original está um nível acima.


In [ ]:
import pandas as pd
import plotly.express as px
import numpy as np

car_data = pd.read_csv('../vehicles_us.csv')

car_data.head()


In [ ]:
car_data.info()


### Estrutura inicial

O conjunto contém **51.525 anúncios** e **13 variáveis**. A inspeção inicial também permite identificar quais colunas possuem valores ausentes e quais tipos de dados foram carregados.


In [ ]:
car_data.describe()


## 2. Análise das variáveis categóricas

Foram verificadas as principais categorias de `condition`, `fuel` e `transmission`.


In [ ]:
car_data['condition'].value_counts()


In [ ]:
car_data['fuel'].value_counts()


In [ ]:
car_data['transmission'].value_counts()


### Outras variáveis categóricas


In [ ]:
car_data['type'].value_counts()


In [ ]:
car_data['model'].value_counts().head(10)


In [ ]:
car_data['paint_color'].value_counts(dropna=False)


## 3. Análise da variável `price`

A variável `price` apresenta uma distribuição assimétrica, com concentração de anúncios em preços relativamente baixos e uma cauda de valores elevados.

Durante a inspeção foram identificados:
- **798 anúncios com preço igual a $1**;
- **1.404 anúncios com preço inferior a $1.000**.

Esses valores foram investigados como possíveis valores extremos. Como não havia evidência suficiente para afirmar que eram erros de registro, eles foram mantidos no conjunto final.


In [ ]:
car_data['price'].value_counts().sort_index().head(10)


In [ ]:
car_data['price'].sort_values(ascending=False).head(10)


In [ ]:
fig = px.histogram(
    car_data[car_data['price'] <= 50000],
    x='price',
    nbins=50,
    title='Distribuição dos preços abaixo de $50.000',
    labels={'price': 'Price'}
)
fig.show()


## 4. Análise de `condition`

A maior parte dos anúncios está nas categorias `excellent` e `good`, seguidas por `like new`, `fair`, `new` e `salvage`.


In [ ]:
fig = px.histogram(
    car_data,
    x='condition',
    title='Distribuição dos anúncios por condição',
    labels={'condition': 'Condition'}
)
fig.show()


## 5. Análise de `days_listed`

A distribuição de `days_listed` é assimétrica, com maior concentração de anúncios nos primeiros dias e uma cauda para períodos mais longos.

Estatísticas observadas:
- média: aproximadamente **39,6 dias**;
- mediana: **33 dias**;
- mínimo: **0 dias**;
- máximo: **271 dias**.


In [ ]:
car_data['days_listed'].describe()


In [ ]:
fig = px.histogram(
    car_data,
    x='days_listed',
    nbins=50,
    title='Distribuição do tempo de permanência dos anúncios',
    labels={'days_listed': 'Days Listed'}
)
fig.show()


In [ ]:
car_data['days_listed'].sort_values().head(10)


In [ ]:
car_data['days_listed'].sort_values(ascending=False).head(10)


## 6. Relação entre ano do modelo e preço

O gráfico de dispersão mostra que veículos mais recentes tendem a apresentar maior concentração de preços, embora existam valores extremos em diferentes anos.


In [ ]:
fig = px.scatter(
    car_data,
    x='model_year',
    y='price',
    title='Relação entre ano do modelo e preço',
    labels={'model_year': 'Model Year', 'price': 'Price'}
)
fig.show()


## 7. Relação entre quilometragem e preço

Existe uma tendência geral de preços menores em veículos com maior quilometragem, embora a relação não seja perfeitamente linear e existam valores extremos.


In [ ]:
fig = px.scatter(
    car_data,
    x='odometer',
    y='price',
    title='Relação entre quilometragem e preço',
    labels={'odometer': 'Odometer', 'price': 'Price'}
)
fig.show()


## 8. Distribuição dos preços por condição


In [ ]:
fig = px.box(
    car_data,
    x='condition',
    y='price',
    title='Distribuição dos preços por condição do veículo',
    labels={'condition': 'Condition', 'price': 'Price'}
)
fig.show()


## 9. Distribuição dos preços por tipo de veículo


In [ ]:
fig = px.box(
    car_data,
    x='type',
    y='price',
    title='Distribuição dos preços por tipo de veículo',
    labels={'type': 'Type', 'price': 'Price'}
)
fig.show()


## 10. Distribuição dos preços por combustível


In [ ]:
fig = px.box(
    car_data,
    x='fuel',
    y='price',
    title='Distribuição dos preços por tipo de combustível',
    labels={'fuel': 'Fuel Type', 'price': 'Price'}
)
fig.show()


## 11. Distribuição dos preços por transmissão


In [ ]:
fig = px.box(
    car_data,
    x='transmission',
    y='price',
    title='Distribuição dos preços por tipo de transmissão',
    labels={'transmission': 'Transmission', 'price': 'Price'}
)
fig.show()


# 12. Identificação e tratamento de valores ausentes

A inspeção inicial mostrou valores ausentes principalmente em `model_year`, `cylinders`, `odometer`, `paint_color` e `is_4wd`.


In [ ]:
car_data.isna().sum()


### `model_year`

A mediana observada foi **2011**. Os valores ausentes foram preenchidos com a mediana.

Também foram identificados anos históricos muito antigos. Para manter a consistência do conjunto utilizado no dashboard, os anos abaixo de **1954** foram limitados a 1954.


In [ ]:
model_year_median = car_data['model_year'].median()
model_year_median


In [ ]:
car_data['model_year'] = car_data['model_year'].fillna(model_year_median)
car_data['model_year'] = car_data['model_year'].clip(lower=1954, upper=2019)


### `cylinders`

A mediana de `cylinders` foi **6** e foi utilizada para preencher os valores ausentes.


In [ ]:
cylinders_median = car_data['cylinders'].median()
cylinders_median


In [ ]:
car_data['cylinders'] = car_data['cylinders'].fillna(cylinders_median)


### `odometer`

A mediana de `odometer` foi **113.000**. Os valores ausentes foram preenchidos com essa mediana.

Também foram identificados **185 registros com quilometragem igual a 0**. Como quilometragem zero não é adequada para veículos usados nesse contexto, esses registros foram substituídos pela mediana.


In [ ]:
odometer_median = car_data['odometer'].median()
odometer_median


In [ ]:
zero_odometer = (car_data['odometer'] == 0).sum()
zero_odometer


In [ ]:
car_data['odometer'] = car_data['odometer'].fillna(odometer_median)
car_data.loc[car_data['odometer'] == 0, 'odometer'] = odometer_median


### `paint_color`

Os valores ausentes de `paint_color` foram substituídos pela categoria `unknown`, preservando os anúncios sem eliminar registros.


In [ ]:
car_data['paint_color'] = car_data['paint_color'].fillna('unknown')


### `is_4wd`

A coluna `is_4wd` funciona como um indicador: `1` representa veículos 4WD e a ausência do valor indica que o veículo não foi identificado como 4WD. Por isso, os valores ausentes foram preenchidos com `0`.


In [ ]:
car_data['is_4wd'] = car_data['is_4wd'].fillna(0)


## 13. Validação após o tratamento

Depois dos tratamentos, verificamos novamente valores ausentes e estatísticas descritivas.


In [ ]:
car_data.isna().sum()


In [ ]:
car_data.info()


In [ ]:
car_data.describe()


### Resultado

Após o tratamento, todas as **51.525 linhas** possuem valores preenchidos nas 13 variáveis.

As principais estatísticas do dataset tratado incluem:
- `price`: média aproximada de **$12.132**;
- `model_year`: mediana **2011**;
- `cylinders`: mediana **6**;
- `odometer`: mediana **113.000**;
- `days_listed`: mediana **33 dias**.


## 14. Exportação do dataset tratado

O dataset final é salvo na raiz do projeto para ser utilizado pelo `app.py`.


In [ ]:
car_data.to_csv('../vehicles_us_clean.csv', index=False)


## Conclusão

A EDA permitiu compreender a estrutura dos anúncios, identificar valores ausentes e extremos, analisar a distribuição das principais variáveis e investigar relações entre características dos veículos e `price`.

O dataset tratado foi utilizado posteriormente no dashboard interativo desenvolvido com Streamlit.
